In [1]:
import pandas as pd
import numpy as np
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

N_VEHICLES = 4
CAPACITY = 130          # 4 x 130 = 520 vs 466 demand — binding but feasible
SOLVE_SECONDS = 60

In [2]:
distance_matrix = np.load('../data/distance_matrix.npy')
sample = pd.read_csv('../data/sample_stops.csv')

BIG_M = 999_999_999
n_unreachable = int((distance_matrix >= BIG_M).sum())
print(f"unreachable pairs: {n_unreachable}")
assert n_unreachable == 0, "some stop pairs have no driving path — graph needs fixing"

unreachable pairs: 0


In [3]:
distance_matrix = distance_matrix.astype(int)
manager = pywrapcp.RoutingIndexManager(
    len(distance_matrix),   # number of locations
    N_VEHICLES,             # number of vehicles
    0                       # depot index
)
routing = pywrapcp.RoutingModel(manager)

In [4]:
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

In [5]:
transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [6]:
print(len(sample))

151


In [7]:
assert 'demand' in sample.columns, "no demand column — run 04 first"

total_demand = int(sample['demand'].sum())
model_vehicles = manager.GetNumberOfVehicles()
model_capacity = model_vehicles * CAPACITY

print(f"model vehicles: {model_vehicles} (expected {N_VEHICLES})")
print(f"demand {total_demand} vs fleet capacity {model_capacity}")

assert model_vehicles == N_VEHICLES, "manager doesn't match N_VEHICLES"
assert total_demand <= model_capacity, "infeasible: not enough capacity"

model vehicles: 4 (expected 4)
demand 466 vs fleet capacity 520


In [8]:
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return int(sample.iloc[from_node]['demand'])
demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

In [9]:
print(sample['demand'].sum())

466


In [10]:
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,
    [CAPACITY] * N_VEHICLES,
    True,
    'Capacity'
)

True

In [11]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = SOLVE_SECONDS
search_parameters.log_search = True

In [12]:
solution = routing.SolveWithParameters(search_parameters)

status = routing.status()
names = {0: 'NOT_SOLVED', 1: 'SUCCESS', 2: 'FAIL_TIMEOUT', 3: 'FAIL_INFEASIBLE', 4: 'INVALID'}
print(f"solver status: {names.get(status, status)}")
assert solution is not None, f"no solution — status {names.get(status, status)}"
print(f"objective: {solution.ObjectiveValue()/1000:.2f} km")

I0000 00:00:1787948063.900055 27082175 search.cc:310] Start search (memory used = 145.19 MB)
I0000 00:00:1787948063.900158 27082175 search.cc:310] Root node processed (time = 0 ms, constraints = 641, memory used = 145.30 MB)
I0000 00:00:1787948064.045565 27082175 search.cc:310] Solution #0 (375848, time = 145 ms, branches = 34, failures = 1, depth = 33, memory used = 146.22 MB, limit = 0%)
I0000 00:00:1787948064.049262 27082175 search.cc:310] Solution #1 (375479, improvement rate = -32.726%/s, maximum = 375848, time = 149 ms, branches = 38, failures = 3, depth = 33, Relocate<1>, neighbors = 1497, filtered neighbors = 1, accepted neighbors = 1, memory used = 146.34 MB, limit = 0%)
I0000 00:00:1787948064.051708 27082175 search.cc:310] Solution #2 (375181, improvement rate = -39.6826%/s, maximum = 375848, time = 151 ms, branches = 43, failures = 5, depth = 33, Relocate<1>, neighbors = 1711, filtered neighbors = 2, accepted neighbors = 2, memory used = 146.62 MB, limit = 0%)
I0000 00:00:17

solver status: SUCCESS
objective: 293.26 km


I0000 00:00:1787948123.833613 27082175 search.cc:310] Solution #521 (294572, improvement rate = -1.03176%/s, minimum = 293258, maximum = 375848, time = 59933 ms, branches = 2920, failures = 1256, depth = 33, TwoOpt, neighbors = 8507571, filtered neighbors = 521, accepted neighbors = 521, memory used = 74.69 MB, limit = 99%)
I0000 00:00:1787948123.890080 27082175 search.cc:310] Finished search tree (time = 59989 ms, branches = 2923, failures = 1289, neighbors = 8559101, filtered neighbors = 521, accepted neigbors = 521, memory used = 74.69 MB)
I0000 00:00:1787948123.890135 27082175 search.cc:310] End search (time = 59990 ms, branches = 2923, failures = 1289, memory used = 74.70 MB, speed = 48 branches/s)


In [13]:
if solution:
    print(solution)
else:
    print("No solution found")

Assignment(Capacity0 (0) | Capacity1 (73) | Capacity2 (64) | Capacity3 (51) | Capacity4 (19) | Capacity5 (8) | Capacity6 (113) | Capacity7 (80) | Capacity8 (43) | Capacity9 (3) | Capacity10 (98) | Capacity11 (28) | Capacity12 (14) | Capacity13 (122) | Capacity14 (27) | Capacity15 (109) | Capacity16 (79) | Capacity17 (13) | Capacity18 (0) | Capacity19 (5) | Capacity20 (125) | Capacity21 (117) | Capacity22 (108) | Capacity23 (61) | Capacity24 (26) | Capacity25 (0) | Capacity26 (0) | Capacity27 (75) | Capacity28 (0) | Capacity29 (3) | Capacity30 (116) | Capacity31 (90) | Capacity32 (5) | Capacity33 (23) | Capacity34 (38) | Capacity35 (13) | Capacity36 (60) | Capacity37 (16) | Capacity38 (31) | Capacity39 (109) | Capacity40 (18) | Capacity41 (71) | Capacity42 (45) | Capacity43 (110) | Capacity44 (22) | Capacity45 (88) | Capacity46 (35) | Capacity47 (52) | Capacity48 (119) | Capacity49 (84) | Capacity50 (76) | Capacity51 (66) | Capacity52 (92) | Capacity53 (57) | Capacity54 (53) | Capacity5

In [14]:
total_distance = 0

for vehicle_id in range(3):
    index = routing.Start(vehicle_id)
    route, route_distance = [], 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(sample.iloc[node]['Address'])
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += distance_matrix[
            manager.IndexToNode(previous_index)
        ][manager.IndexToNode(index)]

    route.append(sample.iloc[manager.IndexToNode(index)]['Address'])
    total_distance += route_distance

    print(f"Driver {vehicle_id + 1}:")
    for stop in route:
        print(f"  -> {stop}")
    print(f"  Route distance: {round(route_distance/1000, 2)} km\n")

print(f"FLEET TOTAL: {round(total_distance/1000, 2)} km")

Driver 1:
  -> 4520 Bullock Farm Rd
  -> 4225 Daly Rd UNIT C
  -> 4235 Daly Rd
  -> 4724 Hargrove Rd UNIT 188
  -> 5024 Departure Dr STE C
  -> 2617 Rowland Rd STE 100
  -> 3209 Gresham Lake Rd STE 114
  -> 6615 Falls Of Neuse Rd
  -> 5886 Faringdon Pl STE 7B
  -> 5878 Faringdon Pl STE 11A
  -> 6411 Lakecrest Dr
  -> 5964 Six Forks Rd UNIT B
  -> 187 Wind Chime Ct STE 101
  -> 134 Mine Lake Ct
  -> 8309 Six Forks Rd STE 101
  -> 8307 Six Forks Rd STE 107
  -> 8382 Six Forks Rd UNIT 104
  -> 8506 Six Forks Rd UNIT 203
  -> 8537 Six Forks Rd STE 600
  -> 9500 Forum Dr
  -> 9207 Baileywick Rd UNIT 101
  -> 8201 Creedmoor Rd
  -> 7201 Creedmoor Rd STE 150
  -> 2301 Stonehenge Dr STE 112
  -> 400 Newton Rd STE 110
  -> 6675 Falls Of Neuse Rd STE 115
  -> 6801 Falls Of Neuse Rd STE 100
  -> 6837 Falls Of Neuse Rd STE 206
  -> 7501 Falls Of Neuse Rd UNIT 200
  -> 8450 Falls Of Neuse Rd UNIT 202
  -> 9111 Litchford Rd
  -> 10010 Falls Of Neuse Rd STE 103
  -> 10405 Falls Of Neuse Rd UNIT 3
  -

In [15]:
solution = routing.SolveWithParameters(search_parameters)

print(f"solver status code: {routing.status()}   (1 = success)")
assert solution is not None, "no solution found"
print(f"objective: {solution.ObjectiveValue()/1000:.2f} km")

I0000 00:00:1787948123.907166 27082175 search.cc:310] Start search (memory used = 79.75 MB)
I0000 00:00:1787948123.907241 27082175 search.cc:310] Root node processed (time = 0 ms, constraints = 641, memory used = 79.75 MB)
I0000 00:00:1787948124.052993 27082175 search.cc:310] Solution #0 (375848, minimum = 293258, time = 145 ms, branches = 2958, failures = 1290, depth = 33, TwoOpt, memory used = 78.92 MB, limit = 0%)
I0000 00:00:1787948124.056684 27082175 search.cc:310] Solution #1 (375479, improvement rate = -32.726%/s, minimum = 293258, maximum = 375848, time = 149 ms, branches = 2962, failures = 1292, depth = 33, Relocate<1>, neighbors = 8560598, filtered neighbors = 522, accepted neighbors = 522, memory used = 78.92 MB, limit = 0%)
I0000 00:00:1787948124.059167 27082175 search.cc:310] Solution #2 (375181, improvement rate = -39.6826%/s, minimum = 293258, maximum = 375848, time = 151 ms, branches = 2967, failures = 1294, depth = 33, Relocate<1>, neighbors = 8560812, filtered neighbo

solver status code: 1   (1 = success)
objective: 293.26 km


I0000 00:00:1787948183.849288 27082175 search.cc:310] Solution #523 (293765, improvement rate = -162.187%/s, minimum = 293258, maximum = 375848, time = 59942 ms, branches = 5856, failures = 2552, depth = 33, TwoOpt, neighbors = 17139345, filtered neighbors = 1044, accepted neighbors = 1044, memory used = 77.64 MB, limit = 99%)
I0000 00:00:1787948183.851807 27082175 search.cc:310] Solution #524 (293805, improvement rate = 6.80816%/s, minimum = 293258, maximum = 375848, time = 59944 ms, branches = 5862, failures = 2554, depth = 33, TwoOpt, neighbors = 17139361, filtered neighbors = 1045, accepted neighbors = 1045, memory used = 77.64 MB, limit = 99%)
I0000 00:00:1787948183.857518 27082175 search.cc:310] Solution #525 (293693, improvement rate = -7.6241%/s, minimum = 293258, maximum = 375848, time = 59950 ms, branches = 5866, failures = 2556, depth = 33, OrOpt<1>, neighbors = 17142306, filtered neighbors = 1046, accepted neighbors = 1046, memory used = 77.66 MB, limit = 99%)
I0000 00:00:1